In [0]:
#NOTEBOOK 2 WITHOUT MODULES WM(05_Gold_Augment_Unit_Test_WM)

from pyspark.sql.functions import col, lower
import time

# ===================================================================================
# CONFIGURATION
# ===================================================================================
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").lower()

SILVER_CATALOG = "hgi"
SILVER_SCHEMA  = "silver"
GOLD_CATALOG   = "hgi"
GOLD_SCHEMA    = "gold"

print("=" * 70)
print("03_Load_Silver_to_Gold_Augment_TestValidation")
print(f"  Customer : {customer_id}")
print("=" * 70)

start_time = time.time()

# ===================================================================================
# STEP 4 — DATA QUALITY VALIDATION
# ===================================================================================
print("\n" + "="*70)
print("STEP 4 — DATA QUALITY VALIDATION")
print("="*70)

dq = {}

# Boolean checks
dq["persons_no_duplicate_emails"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM (
        SELECT mk_email, COUNT(*) AS c
        FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons
        GROUP BY mk_email HAVING c > 1)
""").collect()[0]["cnt"] == 0

dq["persons_email_is_lowercase"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons
    WHERE mk_email != lower(mk_email)
""").collect()[0]["cnt"] == 0

dq["persons_status_not_null"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons
    WHERE mk_status IS NULL
""").collect()[0]["cnt"] == 0

dq["companies_no_duplicate_domains"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM (
        SELECT mk_domain, COUNT(*) AS c
        FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies
        GROUP BY mk_domain HAVING c > 1)
""").collect()[0]["cnt"] == 0

dq["companies_domain_is_lowercase"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies
    WHERE mk_domain != lower(mk_domain)
""").collect()[0]["cnt"] == 0

dq["companies_status_not_null"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies
    WHERE mk_status IS NULL
""").collect()[0]["cnt"] == 0

# Coverage metrics
total_persons   = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons").collect()[0]["cnt"]
ok_persons      = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons WHERE mk_status='ok'").collect()[0]["cnt"]
total_companies = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies").collect()[0]["cnt"]
ok_companies    = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies WHERE mk_status='ok'").collect()[0]["cnt"]
total_contacts  = spark.sql(f"SELECT COUNT(*) AS cnt FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all").collect()[0]["cnt"]
contacts_w_co   = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations WHERE company__name IS NOT NULL").collect()[0]["cnt"]

dq["persons_enrichment_coverage_pct"]            = round((ok_persons / total_persons * 100) if total_persons > 0 else 0, 2)
dq["companies_enrichment_coverage_pct"]          = round((ok_companies / total_companies * 100) if total_companies > 0 else 0, 2)
dq["contacts_computations_company_coverage_pct"] = round((contacts_w_co / total_contacts * 100) if total_contacts > 0 else 0, 2)

# Priority 1 rate check
priority1_total = spark.sql(f"""
    SELECT COUNT(DISTINCT lower(ca.email)) AS cnt
    FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
    JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts uc ON ca.id = uc.id
    WHERE ca.email IS NOT NULL
      AND uc.data_timestamp >= date_add(current_date(), -7)
""").collect()[0]["cnt"]

priority1_ok = spark.sql(f"""
    SELECT COUNT(DISTINCT lower(ca.email)) AS cnt
    FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
    JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts uc ON ca.id = uc.id
    JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON lower(ca.email) = lower(p.mk_email)
    WHERE ca.email IS NOT NULL
      AND uc.data_timestamp >= date_add(current_date(), -7)
      AND p.mk_status = 'ok'
""").collect()[0]["cnt"]

priority1_rate = round((priority1_ok / priority1_total * 100) if priority1_total > 0 else 0, 2)
overall_rate   = dq["persons_enrichment_coverage_pct"]

dq["priority1_enrichment_rate_pct"]       = priority1_rate
dq["priority1_rate_exceeds_overall_rate"] = priority1_rate >= overall_rate

# DQ Report
THRESHOLDS = {
    "persons_enrichment_coverage_pct":             60.0,
    "companies_enrichment_coverage_pct":           70.0,
    "contacts_computations_company_coverage_pct":  50.0,
    "priority1_enrichment_rate_pct":               overall_rate,
}

print("\n  DATA QUALITY REPORT")
print("  " + "-"*65)
all_passed = True
for check, result in dq.items():
    if isinstance(result, bool):
        status = "PASS" if result else "FAIL"
        if not result: all_passed = False
    else:
        threshold = THRESHOLDS.get(check, 0)
        flag = "OK" if result >= threshold else "BELOW THRESHOLD"
        status = f"{result}%  (min: {threshold}%)  {flag}"
        if result < threshold: all_passed = False
    print(f"  {check:<52}: {status}")
print("  " + "-"*65)
print("  All checks passed!" if all_passed else "  Some checks need review.")



# ===================================================================================
# CDC / CDF VALIDATION — Inline (no module)
# ===================================================================================
print("\n" + "="*70)
print("CDC / CDF VALIDATION — Verifying CDF on gold tables")
print("="*70)

cdf_results = {}
try:
    for table in ["persons", "companies", "contacts_computations"]:
        result = spark.sql(f"""
            SHOW TBLPROPERTIES {GOLD_CATALOG}.{GOLD_SCHEMA}.{table}
        """).filter("key = 'delta.enableChangeDataFeed'").collect()

        if result and result[0]["value"].lower() == "true":
            cdf_results[table] = "ENABLED"
            print(f"  {table}: CDF ENABLED")
        else:
            cdf_results[table] = "DISABLED"
            print(f"  {table}: CDF DISABLED")

    all_cdf_enabled = all(v == "ENABLED" for v in cdf_results.values())
    print(f"\n  CDF STATUS: {'ALL GOLD TABLES ENABLED' if all_cdf_enabled else 'WARNING — SOME TABLES DISABLED'}")

except Exception as cdc_e:
    print(f"  WARNING: CDC validation failed: {str(cdc_e)}")

end_time    = time.time()
duration_ms = round((end_time - start_time) * 1000, 2)

# ===================================================================================
# AUDIT LOGGER — COMMENTED (uncomment for production)
# ===================================================================================
# ==================================================================================
# PRODUCTION AUDIT LOGGING — Currently Commented for Dev/Test Environment
# ==================================================================================
# When you uncomment this section, it will:
#   1. Write DQ check results to hgi.gold.audit_logs table
#   2. Capture CDC validation status in the audit record
#   3. Send email alert (SUCCESS if all_passed, DQ_CHECK_FAILED if not)
#   4. Give you full traceability of this test run
#
# HOW TO ACTIVATE:
#   1. Uncomment the %run and log_audit() lines below
#   2. Replace "your_email@company.com" with your actual email
#   3. Run through Databricks Jobs for full stats capture
# ==================================================================================
#
# %run ./utilities/audit_logger.py
# initialize_audit_tables()
#
# log_audit(
#     job_name         = "03_Load_Silver_to_Gold_Augment_TestValidation",
#     customer_id      = customer_id,
#     status           = "success" if all_passed else "failure",
#     alert_type       = "SUCCESS" if all_passed else "DQ_CHECK_FAILED",
#     error_type       = None if all_passed else "DQ_THRESHOLD_NOT_MET",
#     error_reason     = None if all_passed else "One or more DQ thresholds not met",
#     layer            = "gold",
#     rows_processed   = total_persons + total_companies,
#     duration_ms      = int(duration_ms),
#     email_on_failure = "your_email@company.com"
# )

print(f"""
=======================================================================
  TEST VALIDATION COMPLETE
  Customer : {customer_id}
  Status   : {"ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED"}
=======================================================================
  hgi.gold.persons              : {total_persons} rows ({ok_persons} ok)
  hgi.gold.companies            : {total_companies} rows ({ok_companies} ok)
  hgi.gold.contacts_computations: {contacts_w_co} rows with company data
  Priority 1 rate               : {priority1_rate}% vs overall {overall_rate}%
  Duration                      : {duration_ms} ms
  CDF Status                    : {"ENABLED ON ALL" if all_cdf_enabled else "CHECK ABOVE"}
=======================================================================
""")

print("STEP 4 COMPLETE.")
